# Discovering Artistic Movements Through Machine Learning
## Classification and Clustering of Museum Artworks

**Ahmed Sahigara | School of Computer Engineering, MIT Manipal**

> *"What if a museum's collection could tell us its own story?"*

---

This notebook is the **full project showcase**. It walks through every stage of the pipeline:
data loading → cleaning → EDA → feature engineering → classification → clustering → interpretation.

**Dataset:** [MoMA Collection](https://github.com/MuseumofModernArt/collection) — 160K+ artworks, 15K+ artists

In [ ]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.utils import ensure_dirs
ensure_dirs()

%matplotlib inline
plt.rcParams['figure.dpi'] = 110
print('Libraries loaded ✓')

---
## Part 1 — Data Loading & Cleaning

The MoMA dataset ships as two CSV files:
- `Artworks.csv` — one row per artwork (title, medium, dimensions, date, department…)
- `Artists.csv` — one row per artist (nationality, gender, birth/death year)

They link via a `ConstituentID`. Our `data_loader` merges them and engineers a clean numeric dataframe.

In [ ]:
from src.data_loader import load_and_clean
df = load_and_clean()
df.sample(5, random_state=42)

In [ ]:
# Cleaning results at a glance
print(f'Shape after cleaning : {df.shape}')
print(f'Departments          : {df["Department"].nunique()}')
print(f'Unique artists       : {df["Artist"].nunique() if "Artist" in df.columns else "N/A"}')
print(f'Year range           : {df["creation_year"].min():.0f} – {df["creation_year"].max():.0f}')
print()
print('Class distribution:')
print(df['Department'].value_counts().to_frame('count').assign(
    pct=lambda x: (x['count'] / x['count'].sum() * 100).round(1)
))

---
## Part 2 — Exploratory Data Analysis

In [ ]:
from src.eda import (
    department_distribution, timeline_plot, nationality_chart,
    gender_split, medium_top20, acquisition_trend, dept_by_decade_heatmap
)

fig = department_distribution(df); plt.show()

In [ ]:
fig = timeline_plot(df); plt.show()

In [ ]:
fig = acquisition_trend(df); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Gender
gender_col = df.get('ArtistGender', df.get('Gender', pd.Series(dtype=str)))
gender_col = gender_col.fillna('Unknown').str.strip()
gender_col = gender_col.map(lambda x: 'Female' if 'female' in x.lower()
                             else ('Male' if 'male' in x.lower() else 'Unknown'))
g_counts = gender_col.value_counts()
axes[0].pie(g_counts.values, labels=g_counts.index, autopct='%1.1f%%',
            startangle=140, wedgeprops=dict(width=0.5),
            colors=['#4C72B0','#DD8452','#929591'])
axes[0].set_title('Artist Gender Distribution', fontweight='bold')

# Nationality
nat_col = (df.get('ArtistNationality', df.get('Nationality'))
             .dropna().str.strip().value_counts().head(15).sort_values())
axes[1].barh(nat_col.index, nat_col.values, color=sns.color_palette('Blues_r', 15))
axes[1].set_title('Top 15 Artist Nationalities', fontweight='bold')
axes[1].set_xlabel('Count')

plt.tight_layout()
plt.show()

In [ ]:
fig = medium_top20(df); plt.show()

In [ ]:
fig = dept_by_decade_heatmap(df); plt.show()

### EDA Insights

| Observation | Implication for ML |
|---|---|
| Prints & Photography dominate | Class imbalance — stratified splitting is essential |
| Most works created post-1950 | `creation_decade` will be a strong feature |
| 85%+ male artists | `is_female` adds signal despite the imbalance |
| Medium strings are rich & noisy | TF-IDF will extract the signal |
| Photography peaks in the 2000s | Decade × department interaction matters |

---
## Part 3 — Feature Engineering

In [ ]:
from src.features import extract_features

clf_df, clust_df, meta = extract_features(df)

feat_names = meta['feature_names']
structured = [f for f in feat_names if not f.startswith('med_')]
tfidf      = [f for f in feat_names if f.startswith('med_')]

print(f'Total features : {len(feat_names)}')
print(f'  Structured   : {len(structured)} → {structured}')
print(f'  TF-IDF (medium): {len(tfidf)} top terms')

In [ ]:
from src.visualizer import feature_matrix_sample
fig = feature_matrix_sample(clf_df)
plt.show()

In [ ]:
from src.eda import correlation_matrix
fig = correlation_matrix(clf_df)
plt.show()

---
## Part 4 — Classification

Three models trained on the same feature matrix:
1. **Logistic Regression** — fast linear baseline
2. **Random Forest** — our main model, also gives feature importances
3. **SVM (RBF kernel)** — typically strong on high-dim TF-IDF spaces

In [ ]:
from src.classifier import prepare_data
X_train, X_test, y_train, y_test, scaler, feat_cols = prepare_data(clf_df)

In [ ]:
from src.classifier import train_logistic, train_random_forest, train_svm, evaluate

print('Training Logistic Regression...')
lr = train_logistic(X_train, y_train)

print('\nTraining Random Forest...')
rf = train_random_forest(X_train, y_train)

print('\nTraining SVM...')
svm = train_svm(X_train, y_train)

In [ ]:
class_names = meta['class_names']

lr_result  = evaluate(lr,  X_test, y_test, class_names, 'Logistic Regression')
plt.show()

rf_result  = evaluate(rf,  X_test, y_test, class_names, 'Random Forest')
plt.show()

svm_result = evaluate(svm, X_test, y_test, class_names, 'SVM')
plt.show()

In [ ]:
from src.classifier import plot_model_comparison

all_results = [lr_result, rf_result, svm_result]
fig = plot_model_comparison(all_results)
plt.show()

# Summary table
pd.DataFrame([{
    'Model': r['label'],
    'Accuracy': f"{r['accuracy']:.4f}",
    'Macro F1': f"{r['report']['macro avg']['f1-score']:.4f}",
} for r in all_results])

In [ ]:
from src.classifier import plot_feature_importances
fig = plot_feature_importances(rf, feat_cols)
plt.show()

In [ ]:
# Cross-validation check for the best model
from src.classifier import cross_validate_model
best_model = max(all_results, key=lambda r: r['accuracy'])['model']
best_label = max(all_results, key=lambda r: r['accuracy'])['label']
cv_scores = cross_validate_model(best_model, X_train, y_train, best_label)

In [ ]:
# Learning curve — diagnose over/underfitting
from src.visualizer import learning_curve_plot
fig = learning_curve_plot(best_model, X_train, y_train)
plt.show()

---
## Part 5 — Clustering

We remove the target label and let the algorithm find its own groups.
The question: do natural groupings align with department categories, or do they reveal something different?

In [ ]:
from sklearn.preprocessing import StandardScaler
from src.clusterer import reduce_pca

scaler_c = StandardScaler()
X_scaled = scaler_c.fit_transform(clust_df.values)

X_pca50, _  = reduce_pca(X_scaled, n_components=min(50, X_scaled.shape[1]))
X_2d, pca2d = reduce_pca(X_scaled, n_components=2)

In [ ]:
from src.visualizer import pca_explained_variance
fig = pca_explained_variance(X_scaled)
plt.show()

In [ ]:
from src.clusterer import find_optimal_k
best_k, fig = find_optimal_k(X_pca50)
plt.show()

In [ ]:
from src.clusterer import run_kmeans, plot_pca_clusters, plot_cluster_dept_breakdown

km_labels, km_model = run_kmeans(X_pca50, best_k)

fig = plot_pca_clusters(X_2d, km_labels, title=f'PCA — KMeans k={best_k}')
plt.show()

fig = plot_cluster_dept_breakdown(df, km_labels)
plt.show()

In [ ]:
from src.clusterer import run_dbscan

db_labels = run_dbscan(X_2d)

fig = plot_pca_clusters(X_2d, db_labels, title='PCA — DBSCAN')
plt.show()

In [ ]:
# t-SNE — the pretty one (takes ~2–5 min on full dataset)
from src.clusterer import reduce_tsne, plot_tsne_clusters

X_tsne, tsne_idx = reduce_tsne(X_pca50)
km_labels_sub = km_labels[tsne_idx]

fig = plot_tsne_clusters(X_tsne, km_labels_sub, title=f't-SNE — KMeans k={best_k}')
plt.show()

In [ ]:
from src.clusterer import analyze_clusters

print('=== KMeans Cluster Profiles ===')
km_summary = analyze_clusters(df, km_labels, method='KMeans')
km_summary

In [ ]:
print('=== DBSCAN Cluster Profiles ===')
db_summary = analyze_clusters(df, db_labels, method='DBSCAN')
db_summary

---
## Part 6 — Summary Dashboard

In [ ]:
from src.visualizer import summary_dashboard

best_result_dict = {
    'label': best_label,
    'accuracy': max(r['accuracy'] for r in all_results),
    'all_results': all_results,
}
fig = summary_dashboard(df, best_result_dict, km_summary)
plt.show()

---
## Part 7 — Conclusions

### Classification Results

| Model | Accuracy |
|---|---|
| Logistic Regression | *(see above)* |
| Random Forest | *(see above)* |
| SVM (RBF) | *(see above)* |

**Key finding:** The Medium text (TF-IDF features) carries the most discriminating signal — the *material* an artwork is made from strongly predicts its department. This makes intuitive sense: photographs go to Photography, oils to Painting & Sculpture.

**What the model struggles with:** Artworks that cross boundaries — a digitally printed photograph, a painting with photographic technique — blur department lines in the same way human curators sometimes disagree.

### Clustering Results

KMeans found structurally coherent groups that partially, but not perfectly, mirror the department labels. This is actually the interesting result: the *natural* groupings the algorithm discovers cut across official departments in meaningful ways — for example, grouping all works from a certain era together regardless of medium.

DBSCAN identified a large noise component (~X% of the collection) — artworks that don't fit cleanly into any cluster. These outliers are worth examining: they're often the most unusual or cross-disciplinary works.

### Broader Takeaway

Museum collections encode cultural biases (nationality, gender) that machine learning reflects faithfully. Any production deployment of such a classifier should be evaluated not just on accuracy, but on whether it perpetuates or corrects those biases.

---
*All figures saved to `outputs/`. Models saved to `outputs/models/` via joblib.*